<a href="https://colab.research.google.com/github/nazarovdiyor/colab_repository/blob/main/CUDA_va_OpenCL_texnologiyalarining_qiyosiy_tahlili_(NAZAROV_DIYORBEK)_CUDA_muhitida_parallel_dasturni_ishlab_chiqish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def vector_add_gpu(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]
def vector_add_cpu(a, b, c):
    for i in range(a.size):
        c[i] = a[i] + b[i]
def main():
    N = 10_000_000
    print(f"Hisoblash hajmi: {N} ta element.")
    A = np.ones(N, dtype=np.float32)
    B = np.ones(N, dtype=np.float32) * 2.0
    C_cpu = np.zeros(N, dtype=np.float32)
    C_gpu = np.zeros(N, dtype=np.float32)

    start_cpu = time.time()
    vector_add_cpu(A, B, C_cpu)
    end_cpu = time.time()
    cpu_time = end_cpu - start_cpu
    print(f"Ketma-ket hisoblash vaqti (CPU): {cpu_time:.5f} soniya")
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array(N, dtype=np.float32)

    threads_per_block = 256
    blocks_per_grid = (N + (threads_per_block - 1)) // threads_per_block
    vector_add_gpu[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
    cuda.synchronize()
    start_gpu = time.time()
    vector_add_gpu[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
    cuda.synchronize()
    end_gpu = time.time()
    d_C.copy_to_host(C_gpu)
    gpu_time = end_gpu - start_gpu

    print(f"Parallel hisoblash vaqti (GPU):  {gpu_time:.5f} soniya")
    if np.allclose(C_cpu, C_gpu):
        print("\nNatijalar to'g'ri: CPU va GPU natijalari mos tushdi!")
    speedup = cpu_time / gpu_time
    print(f"SAMARADORLIK KO'RSATKICHI (Tezlashish): GPU algoritmi CPU'ga nisbatan {speedup:.2f} marta tezroq ishladi.")
if __name__ == '__main__':
    main()

Hisoblash hajmi: 10000000 ta element.
Ketma-ket hisoblash vaqti (CPU): 7.51832 soniya
Parallel hisoblash vaqti (GPU):  0.00062 soniya

Natijalar to'g'ri: CPU va GPU natijalari mos tushdi!
SAMARADORLIK KO'RSATKICHI (Tezlashish): GPU algoritmi CPU'ga nisbatan 12203.61 marta tezroq ishladi.
